In [5]:
import json
import re

In [7]:
metrics_path = "../data/apple_2023_financial_metrics.json"
calculation_path = "../data/apple_2023_financial_calculations.json"

with open(metrics_path,"r") as f:
    metrics = json.load(f)

with open(calculation_path,"r") as f:
    calculations = json.load(f)

In [8]:
def get_value(metric_key, year):
    return metrics[metric_key]["values"][str(year)]


def percentage_change(current, previous):
    return ((current - previous) / previous) * 100


def margin(part, total):
    return (part / total) * 100


def format_money(value):
    return f"${value:,.0f} million"


def format_percent(value):
    return f"{value:.2f}%"

In [9]:
def detect_calculation_intent(question):
    q = question.lower()

    if "net profit margin" in q:
        return "net_profit_margin"

    if "gross margin" in q and "%" in q:
        return "gross_margin_percent"

    if "operating margin" in q:
        return "operating_margin"

    if "r&d" in q or "research and development" in q:
        if "%" in q or "percentage" in q:
            return "rd_as_percent_of_sales"
        return "research_and_development"

    if "revenue growth" in q or "sales growth" in q:
        return "revenue_growth"

    if "compare" in q and "net sales" in q:
        return "net_sales_comparison"

    if "liabilities to assets" in q:
        return "liabilities_to_assets"

    if "cash to assets" in q:
        return "cash_to_assets"

    return "rag_only"

In [10]:
def calculate_answer(question):
    intent = detect_calculation_intent(question)

    if intent == "net_profit_margin":
        revenue = get_value("total_net_sales", 2023)
        net_income = get_value("net_income", 2023)
        result = margin(net_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's net profit margin in 2023 was {format_percent(result)}.",
            "calculation": f"Net profit margin = Net income / Total net sales × 100 = {format_money(net_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "net_income_2023": net_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "gross_margin_percent":
        revenue = get_value("total_net_sales", 2023)
        gross_margin = get_value("gross_margin", 2023)
        result = margin(gross_margin, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's gross margin percentage in 2023 was {format_percent(result)}.",
            "calculation": f"Gross margin % = Gross margin / Total net sales × 100 = {format_money(gross_margin)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "gross_margin_2023": gross_margin,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "operating_margin":
        revenue = get_value("total_net_sales", 2023)
        operating_income = get_value("operating_income", 2023)
        result = margin(operating_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's operating margin in 2023 was {format_percent(result)}.",
            "calculation": f"Operating margin = Operating income / Total net sales × 100 = {format_money(operating_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "operating_income_2023": operating_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "rd_as_percent_of_sales":
        revenue = get_value("total_net_sales", 2023)
        rd = get_value("research_and_development", 2023)
        result = margin(rd, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's R&D expense as a percentage of sales in 2023 was {format_percent(result)}.",
            "calculation": f"R&D as % of sales = Research and development / Total net sales × 100 = {format_money(rd)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "research_and_development_2023": rd,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "revenue_growth":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased by {format_percent(abs(pct_change))} in 2023 compared with 2022.",
            "calculation": f"Revenue growth = (2023 net sales - 2022 net sales) / 2022 net sales × 100 = ({format_money(revenue_2023)} - {format_money(revenue_2022)}) / {format_money(revenue_2022)} × 100 = {format_percent(pct_change)}",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "net_sales_comparison":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased from {format_money(revenue_2022)} in 2022 to {format_money(revenue_2023)} in 2023.",
            "calculation": f"Difference = {format_money(revenue_2023)} - {format_money(revenue_2022)} = {format_money(change)}. Percentage change = {format_percent(pct_change)}.",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "liabilities_to_assets":
        liabilities = get_value("total_liabilities", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(liabilities, assets)

        return {
            "intent": intent,
            "answer": f"Apple's liabilities-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Liabilities to assets = Total liabilities / Total assets × 100 = {format_money(liabilities)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "total_liabilities_2023": liabilities,
                "total_assets_2023": assets
            }
        }

    if intent == "cash_to_assets":
        cash = get_value("cash_and_cash_equivalents", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(cash, assets)

        return {
            "intent": intent,
            "answer": f"Apple's cash-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Cash to assets = Cash and cash equivalents / Total assets × 100 = {format_money(cash)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "cash_and_cash_equivalents_2023": cash,
                "total_assets_2023": assets
            }
        }

    return None

In [23]:
import pandas as pd
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from sentence_transformers import SentenceTransformer,util

In [26]:
from sentence_transformers import SentenceTransformer

In [27]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2583.15it/s]


In [35]:
import chromadb
chroma_client = chromadb.PersistentClient(
    path="D:/sudhendra/learning projects/GraphRAG/vector_db"
)
collection = chroma_client.get_collection(
    name="apple_2023_10k"
)

In [36]:
collection.count()

824

In [37]:
def rewrite_financial_query(query):
    query_lower = query.lower()

    if "total net sales" in query_lower or "net sales" in query_lower:
        return "Total net sales 383,285 394,328 365,817 Apple 2023 2022 2021"

    return query

In [38]:
def hybrid_retrieval(query, n_text=5, n_tables=5):
    retrieval_query = rewrite_financial_query(query)
    query_embedding = embedding_model.encode(retrieval_query).tolist()

    text_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_text,
        where={"type": "text_chunk"}
    )

    table_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_tables,
        where={"type": "table_row"}
    )

    retrieved = []

    for i in range(len(table_results["documents"][0])):
        retrieved.append({
            "source_type": "table_row",
            "text": table_results["documents"][0][i],
            "page": table_results["metadatas"][0][i]["page"],
            "distance": table_results["distances"][0][i]
        })

    for i in range(len(text_results["documents"][0])):
        retrieved.append({
            "source_type": "text_chunk",
            "text": text_results["documents"][0][i],
            "page": text_results["metadatas"][0][i]["page"],
            "distance": text_results["distances"][0][i]
        })

    return retrieved

In [39]:
def build_hybrid_context(chunks):
    context = ""

    for i, chunk in enumerate(chunks, start=1):
        context += f"\n[Source {i} | Page {chunk['page']} | Type: {chunk['source_type']}]\n"
        context += chunk["text"]
        context += "\n"

    return context

In [40]:
import ollama

In [44]:
def answer_financial_question(question):
    calculation_result = calculate_answer(question)

    chunks = hybrid_retrieval(question, n_text=5, n_tables=5)
    context = build_hybrid_context(chunks)

    if calculation_result:
        prompt = f"""
You are a financial report assistant.

Use the deterministic calculation result as the primary answer.
Use the retrieved context only to support explanation and source references.

Question:
{question}

Deterministic calculation result:
{json.dumps(calculation_result, indent=4)}

Retrieved context:
{context}

Write a clean final answer in this format:

### Answer
[Clear direct answer]

### Values Used
[List values used]

### Calculation
[Show formula and calculation]

### Interpretation
[Brief financial interpretation]

### Sources
[Mention source pages/types from context if available]
"""
    else:
        prompt = f"""
You are a financial report assistant.

Answer the user's question using ONLY the retrieved Apple 2023 Form 10-K context.

Question:
{question}

Retrieved context:
{context}

Rules:
1. Do not invent numbers.
2. If the answer needs calculation but values are unavailable, say context is insufficient.
3. Use table rows for numbers and text chunks for explanation.
4. Include source references.

Answer:
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": 0,
            "top_p": 0.2
        }
    )

    return response["message"]["content"]

In [45]:
print(answer_financial_question(
    "What was Apple's net profit margin in 2023?"
))

 ### Answer
Apple's net profit margin in 2023 was 25.31%.

### Values Used
- Net income: $96,995 million (Source 8)
- Total net sales: $383,285 million (Source 8)

### Calculation
Net profit margin = Net income / Total net sales * 100 = $96,995 million / $383,285 million * 100 = 25.31%

### Interpretation
This indicates that for every dollar of revenue generated in 2023, Apple kept approximately 25 cents as net profit.

### Sources
- Source 8 (Page 31): Consolidated Statements of Operations


In [46]:
print(answer_financial_question(
    "Compare Apple's 2023 and 2022 total net sales."
))

 ### Answer
Apple's total net sales decreased from $394,328 million in 2022 to $383,285 million in 2023.

### Values Used
- Total net sales for 2023: $383,285 million (Source 1, Source 2, Source 3, Source 4)
- Total net sales for 2022: $394,328 million (Source 1, Source 2, Source 3, Source 4)

### Calculation
Difference = $383,285 million - $394,328 million = $-11,043 million. Percentage change = (-11,043 million / $394,328 million) * 100% = -2.80%.

### Interpretation
Apple's total net sales decreased by approximately 2.80% from 2022 to 2023. This decrease could be due to various factors such as competition, product life cycles, and consumer preferences. (Source 10)


In [47]:
print(answer_financial_question(
    "What was Apple's R&D as a percentage of sales in 2023?"
))

 ### Answer
Apple's R&D expense as a percentage of sales in 2023 was 7.80%.

### Values Used
- Research and development: $29,915 million
- Total net sales: $383,285 million

### Calculation
R&D as % of sales = Research and development / Total net sales * 100

### Interpretation
This means that Apple spent approximately 7.80% of its total net sales on research and development in 2023. This indicates a significant investment in innovation to maintain competitiveness in the rapidly evolving technology market.

### Sources
- [Source 1, Page 76, Type: table_row]
- [Source 8, Page 51, Type: text_chunk]
